# Final Submission 

- model-vs-target comparison
- danceability results
includes best model

In [14]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)


In [15]:
outputs_dir = Path("outputs")
baseline_path = outputs_dir / "baseline_danceability_metrics.csv"
regularized_test_path = outputs_dir / "regularized_test_results.csv"

if not baseline_path.exists() or not regularized_test_path.exists():
    raise FileNotFoundError(
        "Please run baseline.ipynb and regularized_models.ipynb first so outputs/ has all comparison files."
    )

baseline_df = pd.read_csv(baseline_path)
regularized_test_df = pd.read_csv(regularized_test_path)

baseline_test = baseline_df[baseline_df["split"] == "test"].copy()
all_test = pd.concat([regularized_test_df, baseline_test], ignore_index=True)
all_test = all_test.drop_duplicates(subset=["target", "model"], keep="last")

all_test.head()


FileNotFoundError: Please run baseline.ipynb and regularized_models.ipynb first so outputs/ has all comparison files.

In [ ]:
comparison_pivot = all_test.pivot(index="target", columns="model", values="r2").sort_index()
comparison_pivot


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
img = ax.imshow(comparison_pivot.values, aspect="auto")
ax.set_xticks(range(len(comparison_pivot.columns)))
ax.set_xticklabels(comparison_pivot.columns, rotation=30, ha="right")
ax.set_yticks(range(len(comparison_pivot.index)))
ax.set_yticklabels(comparison_pivot.index)
ax.set_title("R² Heatmap: Model vs Target")
fig.colorbar(img, ax=ax, label="R²")
plt.tight_layout()
plt.show()


In [ ]:
danceability_rows = all_test[all_test["target"] == "danceability"].sort_values("r2", ascending=False)

display(danceability_rows[["model", "mse", "mae", "r2"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(danceability_rows["model"], danceability_rows["r2"])
axes[0].set_title("Danceability R²")
axes[0].set_ylabel("R²")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(danceability_rows["model"], danceability_rows["mae"])
axes[1].set_title("Danceability MAE")
axes[1].set_ylabel("MAE")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
best_by_target = (
    all_test.sort_values("r2", ascending=False)
    .groupby("target", as_index=False)
    .first()
    .sort_values("r2", ascending=False)
)

best_by_target


In [ ]:
final_dir = Path("outputs/final")
final_dir.mkdir(parents=True, exist_ok=True)

all_test.to_csv(final_dir / "all_test_model_results.csv", index=False)
best_by_target.to_csv(final_dir / "best_model_by_target.csv", index=False)
comparison_pivot.to_csv(final_dir / "r2_comparison_pivot.csv")

print("Saved final tables to outputs/final/")


In [ ]:
print("Final summary (for report):")
for _, row in best_by_target.iterrows():
    print(
        f"- {row['target']}: best model = {row['model']} "
        f"(R²={row['r2']:.3f}, MAE={row['mae']:.3f}, MSE={row['mse']:.4f})"
    )
